# EEG · 00 · Dataset inspection — THINGS-EEG2
**Question:** is the EEG dataset correctly loaded and trial↔image↔target aligned?

Only *reads* the data through `src.data` — no training. EEG is a temporal multichannel signal `[C, T]` (vs the fMRI spatial vector `[V]`).

In [ ]:
# Run from the PROJECT ROOT so relative paths (configs/, data/, outputs/)
# resolve exactly like the scripts do.
import sys, os
_root = os.getcwd()
while _root != os.path.dirname(_root):
    if os.path.isdir(os.path.join(_root, 'src')) and os.path.isdir(os.path.join(_root, 'configs')):
        break
    _root = os.path.dirname(_root)
os.chdir(_root); sys.path.insert(0, _root)
print('project root:', os.getcwd())
import numpy as np
import matplotlib.pyplot as plt
from src.utils import load_config
from src.data import build_datamodule
cfg = load_config('configs/EEG/exp01_eeg_to_clip.yaml')
# Switch montage/subject if you like: cfg['dataset']['channels'] = 63
cfg['dataset']['root_dir']

In [ ]:
dm = build_datamodule(cfg).prepare()
subj = dm.subjects[0]
print('Subjects        :', dm.subjects)
print('Signal (C, T)   :', dm.signal_shape)
print('Trial aggregation:', dm.trial_aggregation)
for s in ('train','val','test'):
    n_trials = len(dm.get_frame(s))
    n_img = len(dm.subject_split_frame(subj, s))
    print(f'{s:>5}: {n_trials:>6} samples (get_frame) | {n_img:>5} unique images')

## Repetitions per image
THINGS-EEG2 shows each image several times (≈4 in train, ≈80 in test). Training uses per-trial samples (augmentation); val/test average over repetitions (SNR).

In [ ]:
reader = dm.subject_reader(subj)
print('train reps/image:', reader.n_reps('train'))
print('test  reps/image:', reader.n_reps('test'))
print('channels        :', reader.ch_names)
print('n_times         :', reader.n_times, '| window (ms):', float(reader.times[0]*1000), '->', float(reader.times[-1]*1000))

## Stimulus image + associated EEG trial

In [ ]:
from src.data import load_image
frame = dm.get_frame('test')  # image-level (mean over reps)
row = frame.iloc[0]
raw = reader.get_signal(row.source, int(row.img_index), int(row.rep))  # [C, T]
norm = dm.normalizer(subj); normed = norm.transform(raw)
t = reader.times * 1000.0
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(load_image(row.image_path)); axes[0].axis('off')
axes[0].set_title(f'stimulus: {row.image_id}', fontsize=9)
for c in range(raw.shape[0]): axes[1].plot(t, raw[c], lw=0.6)
axes[1].set_title('EEG (mean over reps), raw'); axes[1].set_xlabel('ms')
for c in range(normed.shape[0]): axes[2].plot(t, normed[c], lw=0.6)
axes[2].set_title('per-channel normalized'); axes[2].set_xlabel('ms')
plt.tight_layout(); plt.show()

## Per-channel normalization (fitted on train only)

In [ ]:
mu, sd = norm.mean, norm.std
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(range(len(mu)), mu); ax[0].set_title('per-channel mean (train)')
ax[0].set_xticks(range(len(mu))); ax[0].set_xticklabels(reader.ch_names, rotation=90, fontsize=6)
ax[1].bar(range(len(sd)), sd); ax[1].set_title('per-channel std (train)')
ax[1].set_xticks(range(len(sd))); ax[1].set_xticklabels(reader.ch_names, rotation=90, fontsize=6)
plt.tight_layout(); plt.show()

**Takeaway:** if channels, time window, repetition counts and the trial↔image mapping look right, the EEG data is correctly loaded and aligned to the visual targets (CLIP/VAE-PCA are precomputed per unique image and gathered by `feat_idx`).